- ## New Parameter Interface for the models

In [155]:
from typing import List, Tuple
import numpy as np
import warnings
from io import StringIO
from copy import deepcopy
from tabulate import tabulate
from priors import Prior, UniformPrior
from functools import singledispatchmethod
from parameter import ParameterHandler
from priors import Prior, UniformPrior

In [156]:
from typing import NoReturn, Union


class ParameterValidator:
    """
    Classe per la gestione della validazione di parametri singoli.
    """

    @staticmethod
    def validate_prior(prior: Prior) -> None:
        if not isinstance(prior, Prior):
            raise TypeError("Pior Must be istance of class Prior")

    @staticmethod
    def validate_name(name: str) -> None:
        """
        Valida il nome del parametro.

        Args:
            name (str): Nome del parametro.

        Raises:
            TypeError: Se il nome non è una stringa.
        """
        if not isinstance(name, str):
            raise TypeError("Il nome del parametro deve essere una stringa!")

    @staticmethod
    def validate_bounds(bounds) -> NoReturn:
        """
        Valida i limiti del parametro.

        Args:
            bounds (Tuple[float, float]): Limiti del parametro.

        Raises:
            TypeError: Se i limiti non sono una tupla di due elementi.
            ValueError: Se i limiti non sono validi.
        """
       
        if len(bounds) != 2:
            raise ValueError("I limiti devono essere una tupla con due elementi.")
        if not bounds[0] <= bounds[1]:
            raise ValueError(
                "Il limite inferiore deve essere minore o uguale al limite superiore."
            )
    
    @staticmethod
    def validate_bounds_array(bounds):     
        if not isinstance(bounds, (list, np.ndarray)):
            raise TypeError("bounds must be in form of list or array")
        
        if not all(lower <= upper for (lower, upper) in bounds):
            raise ValueError(
                "Il limite inferiore deve essere minore o uguale al limite superiore."
            )
    
    

    @staticmethod
    def validate_value_in_bounds(value: float, bounds: Tuple[float, float]) -> None:
        """
        Valida che il valore del parametro sia entro i limiti specificati.

        Args:
            value (float): Valore del parametro.
            bounds (Tuple[float, float]): Limiti del parametro.

        Raises:
            TypeError: Se il valore non è un numero.
            ValueError: Se il valore è fuori dai limiti.
        """
        if not isinstance(value, (int, float)):
            raise TypeError("Value must be of type Number, not ", type(value))
        
        if not bounds[0] <= value <= bounds[1]:
            raise ValueError(f"Il valore {value} è fuori dai limiti {bounds}")
        
    @staticmethod
    def validate_value_in_bounds_array(value: np.ndarray, bounds: list[tuple]) -> None:
        """
        Valida che il valore del parametro sia entro i limiti specificati.

        Args:
            value (float): Valore del parametro.
            bounds (Tuple[float, float]): Limiti del parametro.

        Raises:
            TypeError: Se il valore non è un numero.
            ValueError: Se il valore è fuori dai limiti.
        """
        if len(value) != len(bounds):
            raise ValueError("Bounds and Value must have the same size")
        if not isinstance(value, np.ndarray):
            raise TypeError("Value must be of type Array, not ", type(value))
        if not all(lower <= val <= upper for val, (lower, upper) in zip(value, bounds)):
            raise ValueError(f"Il valore {value} è fuori dai limiti {bounds}")
        
        

    @staticmethod
    def validate_frozen(is_true: bool) -> None:
        """
        Valida lo stato di congelamento del parametro.

        Args:
            is_true (bool): Stato di congelamento del parametro.

        Raises:
            TypeError: Se il valore non è un booleano.
        """
        if not isinstance(is_true, bool):
            raise TypeError(
                f'Il valore di "frozen" può essere solo True o False, hai fornito {is_true, type(is_true)}'
            )

    @staticmethod
    def validate_description(strg: str) -> None:
        """
        Valida la descrizione del parametro.

        Args:
            strg (str): Descrizione del parametro.

        Raises:
            TypeError: Se la descrizione non è una stringa.
        """
        if not isinstance(strg, str):
            raise TypeError("Description must be a string!")
        
    @singledispatchmethod
    @staticmethod
    def choose_correct_type(value) -> NotImplementedError:
        "helper function to get the correct array type, the number of dimensions and the type"
        return NotImplementedError()
    
    @choose_correct_type.register
    def _(value:list) -> Tuple[np.ndarray, int]:# -> NDArray:
        val = np.array(value)
        return val, val.ndim
    
    @choose_correct_type.register
    def _(value:np.ndarray)-> Tuple[np.ndarray, int]:
        return value, value.ndim
    @choose_correct_type.register
    def _(value:Union[float, int])-> Tuple[float, int]:
        return value, 1
    @choose_correct_type.register
    def _(value:Union[dict, set]) -> NoReturn:
        raise TypeError(f"{type(value)} is not a supported type for a parameter")
        

    # NOTE disabled fot his test
    # @staticmethod
    # def validate_constrain(value: "Constrain") -> None:
    #     if not isinstance(value, Constrain):
    #         raise TypeError("Parameter constrain must be of type Constrain")


- ## COSE DA CAMBIARE PER ACCOMODARE UN PARAMETRO N DIMENSIONALE
- modificare i bounds => aggiungere funzione check bounds per applicrae correttamente i bounds
- parameter validator deve sapere se ha un parametro a ndim o no
- value.setter deve cambiare il numero di dimensioni del parametro
- self.ndim da creare
- la creazione iniziale del prior deve essere cambiata

In [157]:
import numpy as np
from typing import Tuple, List
import warnings
from copy import deepcopy
from io import StringIO
from tabulate import tabulate

# Supponiamo che esistano queste entità in altri moduli:
# - ParameterValidator con i metodi: choose_correct_type, validate_name,
#   validate_bounds, validate_value_in_bounds, validate_frozen, validate_description, validate_prior.
# - UniformPrior con il metodo _get_str().
# - ParameterHandler


# Classe base: gestisce solo gli attributi comuni
class Parameter:
    def __new__(cls, *args, **kwargs):
        """
        Factory method: estrae 'name' e 'value' dagli argomenti passati e, se si
        sta creando direttamente una istanza di Parameter, sceglie la sottoclasse
        adeguata (ArrayParameter se value è un array o una lista, altrimenti FloatParameter).
        """
        # Proviamo a estrarre 'name' e 'value' dagli argomenti (sia posizionali che keyword)
        try:
            # Se sono passati come keyword, li usiamo; altrimenti assumiamo il loro ordine
            name = kwargs.get("name", args[0])
            value = kwargs.get("value", args[1])
        except IndexError:
            raise TypeError("Missing required arguments 'name' and 'value'")

        if cls is Parameter:
            if isinstance(value, (np.ndarray, list)):
                cls = ArrayParameter
            else:
                cls = FloatParameter
        return super().__new__(cls)

    def __init__(
        self,
        name: str,
        value,  # value e bounds non vengono gestiti qui
        frozen: bool = False,
        bounds: Tuple[float, float] = (-float("inf"), float("inf")),
        description: str = "",
        prior=None,
    ):
        self._name = name
        self._frozen = frozen
        self._description = description
        self._handler: List["ParameterHandler"] = []
        self._chached_properties = ["value", "bounds", "frozen"]
        # Flag legacy
        self._constrain = False
        self._is_tied = False

        if prior is None:
            self._prior = UniformPrior(-float("inf"), float("inf"))
        else:
            self._prior = prior

    def _update_handler_cache(self) -> None:
        if self._handler:
            for handler in self._handler:
                handler._update_cache()

    @property
    def is_free(self) -> bool:
        return not (self.frozen or self.is_tied)

    @property
    def is_tied(self) -> bool:
        return self._is_tied

    @property
    def has_constrain(self) -> bool:
        return self._constrain

    @has_constrain.setter
    def has_constrain(self, value: bool):
        self._constrain = value
        self._update_handler_cache()

    @property
    def handler(self) -> List["ParameterHandler"]:
        return self._handler

    @property
    def prior(self):
        return self._prior

    @prior.setter
    def prior(self, value) -> None:
        ParameterValidator.validate_prior(value)
        self._prior = value

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, new_name: str) -> None:
        ParameterValidator.validate_name(new_name)
        self._name = new_name
        self._update_handler_cache()

    @property
    def frozen(self) -> bool:
        return self._frozen

    @frozen.setter
    def frozen(self, is_true: bool) -> None:
        ParameterValidator.validate_frozen(is_true)
        self._frozen = is_true
        self._update_handler_cache()

    @property
    def description(self) -> str:
        return self._description

    @description.setter
    def description(self, new_description: str) -> None:
        ParameterValidator.validate_description(new_description)
        self._description = new_description

    def copy(self) -> "Parameter":
        return deepcopy(self)

    def __len__(self) -> int:
        # La lunghezza verrà definita nelle sottoclassi
        return 0

    def __iter__(self):
        yield self._name
        yield self._value
        yield self._frozen
        yield self._bounds
        yield self._description

    def __getitem__(self, key):
        return getattr(self, key)

    def __setitem__(self, key, value) -> None:
        setattr(self, key, value)

    def __str__(self) -> str:
        # Implementazione "di default" (sovrascritta nelle sottoclassi)
        return f"Parameter(name={self.name}, value={self.value})"

    def __call__(self, value):
        return self.prior(value)


# Sottoclasse per parametri con valore float.
class FloatParameter(Parameter):
    def __init__(
        self,
        name: str,
        value: float,
        frozen: bool = False,
        bounds: Tuple[float, float] = (-float("inf"), float("inf")),
        description: str = "",
        prior=None,
    ):
        if not isinstance(value, (int, float)):
            raise TypeError(
                "FloatParameter: il valore deve essere un numero (int o float)."
            )
        # Chiamo il costruttore della base (passando None per value e bounds, gestiti qui sotto)
        super().__init__(name, None, frozen, None, description, prior)
        # Gestione specifica del valore float:
        self._value, self._ndim = ParameterValidator.choose_correct_type(value)
        ParameterValidator.validate_value_in_bounds(value, bounds)
        self._bounds = bounds

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new):
        ParameterValidator.validate_value_in_bounds(new, self.bounds)
        val, dim = ParameterValidator.choose_correct_type(new)
        self._value, self._ndim = val, dim
        self._update_handler_cache()

    @property
    def bounds(self):
        return self._bounds

    @bounds.setter
    def bounds(self, new):
        if self.frozen:
            warnings.warn(
                f"Parameter {self.name} is frozen, new bounds will be ignored!"
            )
            return
        ParameterValidator.validate_bounds(new)
        ParameterValidator.validate_value_
        self._bounds = new
        self._update_handler_cache()

    def __str__(self) -> str:
        buffer = StringIO()
        buffer.write(f"PARAM NAME: {self.name}\n")
        field_names = ["NAME", "VALUE", "IS-FREE", "PRIOR", "DESCR:"]
        value_str = f"{self._value:.5g}"
        froz_str = "Yes" if self.is_free else "No"
        prior_str = self.prior._get_str()
        table_data = [[self.name, value_str, froz_str, prior_str, self.description]]
        table = tabulate(
            table_data,
            headers=field_names,
            tablefmt="plain",
            showindex="always",
            colalign=("left",),
        )
        buffer.write(table + "\n")
        return buffer.getvalue()

    def __len__(self):
        # Per un parametro scalare la lunghezza è 1.
        return 1

    def __call__(self, value):
        return self.prior(value)


# Sottoclasse per parametri con valore array (numpy.ndarray o lista).
class ArrayParameter(Parameter):
    def __init__(
        self,
        name: str,
        value: np.ndarray,
        frozen: bool = False,
        bounds: Tuple[float, float] = None,
        description: str = "",
        prior=None,
    ):
        if not isinstance(value, (np.ndarray, list)):
            raise TypeError(
                "ArrayParameter: il valore deve essere un numpy.ndarray o una lista."
            )
        # Se value è una lista, la converto in array NumPy
        if isinstance(value, list):
            value = np.array(value)
        super().__init__(name, None, frozen, None, description, prior)
        # Gestione specifica del valore array:
        self._value, self._ndim = ParameterValidator.choose_correct_type(value)
        ParameterValidator.validate_value_in_bounds(value, bounds, self._ndim)
        if bounds is None:
            # Impostiamo dei bounds di default: una coppia (−∞, ∞) per ogni elemento
            self._bounds = [(-np.inf, np.inf) for _ in range(np.size(self._value))]
        else:
            self._bounds = bounds

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new):
        ParameterValidator.validate_value_in_bounds(new, self.bounds, self._ndim)
        val, dim = ParameterValidator.choose_correct_type(new)
        self._value, self._ndim = val, dim
        self._update_handler_cache()

    @property
    def bounds(self):
        return self._bounds

    @bounds.setter
    def bounds(self, new):
        if self.frozen:
            warnings.warn(
                f"Parameter {self.name} is frozen, new bounds will be ignored!"
            )
            return
        ParameterValidator.validate_bounds(new, self._ndim)
        ParameterValidator.validate_value_in_bounds(self._value, new, self._ndim)
        self._bounds = new
        self._update_handler_cache()

    def __str__(self) -> str:
        buffer = StringIO()
        buffer.write(f"ArrayParameter: {self.name}\n")
        field_names = ["NAME", "VALUE", "IS-FREE", "PRIOR", "DESCR:"]
        # Se l'array contiene più di 3 elementi, mostro solo i primi 3 seguiti da "..."
        if self._value.size > 3:
            first_three = list(self._value.flat)[:3]
            value_str = "[" + ", ".join(f"{x:.5g}" for x in first_three) + ", ...]"
        else:
            value_str = "[" + ", ".join(f"{x:.5g}" for x in self._value.flat) + "]"
        froz_str = "Yes" if self.is_free else "No"
        prior_str = self.prior._get_str()
        table_data = [[self.name, value_str, froz_str, prior_str, self.description]]
        table = tabulate(
            table_data,
            headers=field_names,
            tablefmt="plain",
            showindex="always",
            colalign=("left",),
        )
        buffer.write(table + "\n")
        return buffer.getvalue()

    def __len__(self):
        # La lunghezza equivale al numero di elementi dell'array.
        return len(self._value)

    def __call__(self, value):
        # Per ogni elemento di un array, applica il prior.
        return [self.prior(v) for v in value.flatten()]

In [158]:
p = Parameter("test", 1)
print(p)
print(p.bounds)
print(len(p))

PARAM NAME: test
    NAME      VALUE  IS-FREE    PRIOR               DESCR:
0   test          1  Yes        Uniform(-inf, inf)

(-inf, inf)
1


In [159]:
p.copy()

TypeError: Missing required arguments 'name' and 'value'